# OpenER GRPO Training Notebook

This notebook is self-contained for training setup:
- creates a dedicated virtual environment under `/workspace/venvs/opener`
- installs all Python dependencies into that venv
- registers a Jupyter kernel named `Python (opener)`
- uses the official `open_er` client for stateful OpenEnv sessions
- defines prompts, rollout logic, rewards, dataset, GRPO config, and trainer inline

Recommended runtime:
- RunPod L4 or A100
- This notebook assumes you will use `vLLM` and the TRL `rollout_func` pattern


## 1. Bootstrap the virtual environment

Run this cell once. It creates `/workspace/venvs/opener`, installs dependencies there, and registers a notebook kernel.

After it finishes:
1. go to `Kernel -> Change Kernel`
2. select `Python (opener)`
3. restart the kernel
4. continue from the next cell


In [ ]:
%%bash
set -euo pipefail

VENV_DIR=/workspace/venvs/opener
PYTHON_BIN="$VENV_DIR/bin/python"

if [ ! -d "$VENV_DIR" ]; then
  python3 -m venv "$VENV_DIR"
fi

"$PYTHON_BIN" -m pip install --upgrade pip
"$PYTHON_BIN" -m pip install -U "trl[vllm]" datasets accelerate transformers huggingface_hub requests ipykernel ipywidgets
"$PYTHON_BIN" -m pip install -U git+https://huggingface.co/spaces/joshaeeee/open-er
"$PYTHON_BIN" -m ipykernel install --user --name opener --display-name "Python (opener)"

echo "Bootstrap complete. Switch the notebook kernel to: Python (opener)"

## 2. Verify the active kernel

This should print `/workspace/venvs/opener/bin/python`.
If it does not, stop and switch kernels before continuing.


In [ ]:
import sys

print(sys.executable)
if "/workspace/venvs/opener/bin/python" not in sys.executable:
    raise RuntimeError(
        "Wrong kernel active. Switch to 'Python (opener)' before running the training cells."
    )


## 3. Optional Hugging Face login

Only needed if you want to push checkpoints or models to the Hub.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()


## 4. Imports

In [ ]:
import json
import math
import os
import re
from dataclasses import dataclass
from typing import Any

from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from trl.experimental.openenv import generate_rollout_completions

from open_er import ERAction, OpenEREnv

os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")


## 5. System prompt and environment config

In [ ]:
BASE_URL = "https://joshaeeee-open-er.hf.space"
DEFAULT_DATASET_PROMPT = "Manage the emergency department safely and efficiently."

VALID_TESTS = [
    "bmp", "blood_culture", "cbc", "ct_abdomen", "ct_chest", "ct_head",
    "ecg", "lactate", "troponin", "urinalysis", "xray"
]
VALID_SPECIALISTS = [
    "cardiology", "critical_care", "neurology", "neurosurgery", "pulmonology", "surgery"
]

SYSTEM_PROMPT = f"""You are an expert emergency department triage and throughput agent operating a simulated ER shift.

Your job is to make safe, high-quality, resource-aware decisions for multiple concurrent patients.

PRIMARY OBJECTIVE
- Protect patient safety first.
- Then improve timeliness, diagnostic quality, and resource stewardship.

IMPORTANT SAFETY PRINCIPLES
- Undertriage is dangerous. If a patient looks unstable, escalate acuity rather than minimizing risk.
- Unsafe discharge is heavily penalized. Do not discharge unstable or insufficiently worked-up patients.
- Critical visible deterioration matters more than throughput.
- Use only the visible information in the observation. Hidden diagnoses are not available to you.

ENVIRONMENT FACTS
- One environment step represents a 5-minute turn in the ER.
- You may act on multiple patients in a single step.
- Beds, CT, labs, and specialists are resource constrained.
- Some patients deteriorate if they wait too long or are mishandled.
- Official tasks: easy_single_critical, medium_evening_rush, hard_capacity_crunch.

AVAILABLE ACTION FIELDS
- patient_id: the visible patient identifier
- assign_bed: true or false
- new_esi: integer 1-5 or null
- order_tests: list of test names
- disposition: \"home\", \"admit\", or null
- call_specialist: specialist name or null

VALID TEST NAMES
- {', '.join(VALID_TESTS)}

VALID SPECIALISTS
- {', '.join(VALID_SPECIALISTS)}

OUTPUT FORMAT
- Output JSON only.
- Do not include markdown, explanations, analysis, or prose.
- The JSON must match this schema exactly:
{{
  \"commands\": [
    {{
      \"patient_id\": \"pt_001_0\",
      \"assign_bed\": false,
      \"new_esi\": null,
      \"order_tests\": [],
      \"disposition\": null,
      \"call_specialist\": null
    }}
  ]
}}

ACTION QUALITY GUIDELINES
- Assign beds to the most unstable or highest-acuity waiting patients first.
- Increase ESI urgency when NEWS2/qSOFA and vitals suggest instability.
- Order complaint-appropriate tests. Avoid wasteful shotgun testing.
- Call specialists when complaint pattern and severity make that consult plausible.
- Admit unstable or high-risk patients; discharge only stable low-acuity patients with enough workup.
- If no action is justified, return {{\"commands\": []}}.
"""

@dataclass
class OpenERTRLConfig:
    base_url: str = BASE_URL
    task_id: str = "easy_single_critical"
    seed_start: int = 11
    max_patients_in_prompt: int = 12
    max_episode_steps: int = 8
    dense_reward_scale: float = 10.0
    dense_reward_weight: float = 0.25
    benchmark_weight: float = 1.0
    system_prompt: str = SYSTEM_PROMPT
    dataset_prompt: str = DEFAULT_DATASET_PROMPT

env_cfg = OpenERTRLConfig()
env_cfg


## 6. Prompting, parsing, and reward helpers

In [ ]:
def prompt_to_text(prompt: Any) -> str:
    if isinstance(prompt, str):
        return prompt
    if isinstance(prompt, list):
        parts = []
        for item in prompt:
            if isinstance(item, dict):
                parts.append(f"{item.get('role', 'user')}: {item.get('content', '')}")
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(prompt)


def render_observation(observation, max_patients: int = 12) -> str:
    if hasattr(observation, "model_dump"):
        observation = observation.model_dump()

    resources = observation["resources"]
    lines = [
        f"task_id: {observation['task_id']}",
        f"difficulty: {observation['difficulty']}",
        f"shift_minute: {observation['shift_minute']}",
        (
            "resources: "
            f"beds_available={resources['beds_available']}/{resources['beds_total']}, "
            f"ct_available_in_min={resources['ct_available_in_min']}, "
            f"lab_queue_length={resources['lab_queue_length']}"
        ),
    ]

    alerts = observation.get("alerts") or []
    if alerts:
        lines.append(f"alerts: {', '.join(alerts)}")

    events = observation.get("events") or []
    if events:
        lines.append(f"recent_events: {' | '.join(events[-3:])}")

    metadata = observation.get("metadata") or {}
    reward_breakdown = metadata.get("reward_breakdown") or {}
    if reward_breakdown:
        lines.append(
            "last_reward_breakdown: " + ", ".join(
                f"{key}={value}" for key, value in reward_breakdown.items()
            )
        )

    lines.append("patients:")
    patients = observation.get("patients") or []
    for patient in patients[:max_patients]:
        vitals = patient["vitals"]
        lines.append(
            "  - "
            f"{patient['patient_id']}: complaint={patient['chief_complaint']}, age={patient['age']}, "
            f"esi={patient['assigned_esi']}, loc={patient['location']}, wait={patient['wait_time_min']}, "
            f"news2={patient['news2_score']}, qsofa={patient['qsofa_score']}, "
            f"hr={vitals['hr']}, sbp={vitals['sbp']}, rr={vitals['rr']}, o2={vitals['o2_sat']}, temp={vitals['temp_c']}, "
            f"tests={sorted((patient.get('completed_tests') or {}).keys())}"
        )

    if len(patients) > max_patients:
        lines.append(f"  - ... {len(patients) - max_patients} more patients omitted")

    return "\n".join(lines)


def make_user_prompt(dataset_prompt: str, observation, step_num: int, cfg: OpenERTRLConfig) -> str:
    observation_text = render_observation(observation, max_patients=cfg.max_patients_in_prompt)
    return (
        f"Episode objective: {dataset_prompt}\n\n"
        f"Current turn: {step_num + 1} / {cfg.max_episode_steps}\n"
        "Decide the next 5-minute ER action bundle.\n\n"
        f"{observation_text}\n\n"
        "Return JSON only."
    )


def build_chat_prompt(tokenizer, system_prompt: str, user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return f"system: {system_prompt}\n\nuser: {user_prompt}\n\nassistant:"


def parse_action_json(text: str) -> ERAction:
    fenced = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    if fenced:
        candidate = fenced.group(1)
    else:
        brace = re.search(r"\{.*\}", text, flags=re.DOTALL)
        candidate = brace.group(0) if brace else ""

    if not candidate:
        return ERAction(commands=[])

    try:
        payload = json.loads(candidate)
        if isinstance(payload, dict) and "action" in payload and isinstance(payload["action"], dict):
            payload = payload["action"]
        return ERAction(**payload)
    except Exception:
        return ERAction(commands=[])


def compute_combined_reward(dense_return: float, benchmark_score: float, cfg: OpenERTRLConfig) -> float:
    dense_component = math.tanh(dense_return / cfg.dense_reward_scale)
    return cfg.benchmark_weight * benchmark_score + cfg.dense_reward_weight * dense_component


def decode_completion(output: dict[str, Any], tokenizer) -> str:
    if output.get("text"):
        return str(output["text"])
    return tokenizer.decode(output["completion_ids"], skip_special_tokens=True)


## 7. Smoke test the live environment

In [ ]:
with OpenEREnv(base_url=env_cfg.base_url).sync() as env:
    result = env.reset(task_id=env_cfg.task_id, seed=env_cfg.seed_start)
    print(result.observation.task_id)
    print(result.observation.difficulty)
    print(len(result.observation.patients))
    print(render_observation(result.observation, max_patients=3))

with OpenEREnv(base_url=env_cfg.base_url).sync() as env:
    result = env.reset(task_id=env_cfg.task_id, seed=env_cfg.seed_start)
    result = env.step(ERAction(commands=[]))
    print(result.reward, result.done)


## 8. Rollout function and reward functions

In [ ]:
def rollout_once(trainer, env, tokenizer, dataset_prompt: str, cfg: OpenERTRLConfig, seed: int):
    result = env.reset(task_id=cfg.task_id, seed=seed)

    prompt_ids = []
    completion_ids = []
    logprobs = []
    step_rewards = []
    parsed_actions = []
    completion_texts = []

    for step_num in range(cfg.max_episode_steps):
        if result.done:
            break

        user_prompt = make_user_prompt(dataset_prompt, result.observation, step_num, cfg)
        prompt_text = build_chat_prompt(tokenizer, cfg.system_prompt, user_prompt)

        rollout_output = generate_rollout_completions(trainer, [prompt_text])[0]
        prompt_ids.extend(rollout_output["prompt_ids"])
        completion_ids.extend(rollout_output["completion_ids"])
        logprobs.extend(rollout_output["logprobs"])

        completion_text = decode_completion(rollout_output, tokenizer)
        action = parse_action_json(completion_text)

        parsed_actions.append(action.model_dump())
        completion_texts.append(completion_text)

        result = env.step(action)
        step_rewards.append(float(result.reward or 0.0))

    state = env.state()
    benchmark_score = float(state.benchmark_score or 0.0)
    dense_return = float(sum(step_rewards))
    combined_reward = compute_combined_reward(dense_return, benchmark_score, cfg)

    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "step_rewards": step_rewards,
        "dense_return": dense_return,
        "benchmark_score": benchmark_score,
        "combined_reward": combined_reward,
        "parsed_actions": parsed_actions,
        "completion_texts": completion_texts,
    }


seed_counter = {"value": env_cfg.seed_start}

def rollout_func(prompts, trainer):
    tokenizer = trainer.processing_class

    prompt_ids_batches = []
    completion_ids_batches = []
    logprobs_batches = []
    dense_returns = []
    benchmark_scores = []
    combined_rewards = []
    parsed_actions = []
    step_reward_traces = []

    for prompt in prompts:
        with OpenEREnv(base_url=env_cfg.base_url).sync() as env:
            seed = seed_counter["value"]
            seed_counter["value"] += 1

            episode = rollout_once(
                trainer=trainer,
                env=env,
                tokenizer=tokenizer,
                dataset_prompt=prompt_to_text(prompt) or env_cfg.dataset_prompt,
                cfg=env_cfg,
                seed=seed,
            )

        prompt_ids_batches.append(episode["prompt_ids"])
        completion_ids_batches.append(episode["completion_ids"])
        logprobs_batches.append(episode["logprobs"])
        dense_returns.append(episode["dense_return"])
        benchmark_scores.append(episode["benchmark_score"])
        combined_rewards.append(episode["combined_reward"])
        parsed_actions.append(episode["parsed_actions"])
        step_reward_traces.append(episode["step_rewards"])

    return {
        "prompt_ids": prompt_ids_batches,
        "completion_ids": completion_ids_batches,
        "logprobs": logprobs_batches,
        "dense_return": dense_returns,
        "benchmark_score": benchmark_scores,
        "combined_reward": combined_rewards,
        "parsed_actions": parsed_actions,
        "step_rewards": step_reward_traces,
    }


def reward_combined(completions, **kwargs):
    rewards = kwargs.get("combined_reward") or []
    if rewards:
        return [float(r) for r in rewards]
    return [0.0] * len(completions)


def reward_benchmark(completions, **kwargs):
    rewards = kwargs.get("benchmark_score") or []
    if rewards:
        return [float(r) for r in rewards]
    return [0.0] * len(completions)


def reward_dense_return(completions, **kwargs):
    rewards = kwargs.get("dense_return") or []
    if rewards:
        return [float(r) for r in rewards]
    return [0.0] * len(completions)


## 9. Dataset

In [ ]:
dataset_prompt = DEFAULT_DATASET_PROMPT
dataset_size = 256
dataset = Dataset.from_dict({"prompt": [dataset_prompt] * dataset_size})
dataset


## 10. GRPO config

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "open-er-grpo"

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    max_steps=100,
    learning_rate=5e-6,
    warmup_steps=10,
    per_device_train_batch_size=1,
    num_generations=4,
    generation_batch_size=4,
    max_completion_length=256,
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.2,
    logging_steps=1,
    report_to="none",
)

grpo_config


## 11. Create the GRPO trainer

In [ ]:
trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=[reward_combined],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)

trainer


## 12. Start training

In [ ]:
trainer_stats = trainer.train()
trainer_stats


## 13. Optional save / push

In [ ]:
# trainer.save_model(OUTPUT_DIR)
# trainer.push_to_hub()
